# SP-004 — Raster Hole Filling / Pocket Filling

Adaptación local de Abeysooriya et al. (2018): un pocket es un componente interior 4-conexo de anclajes legales para pieza y orientación; puede existir sin hueco físico cerrado. Se valida con `SheetLayout.place()` y se recurre a FF si no aparece.

Configuración: diagnósticos 1 mm/5°/margen 1 mm/separación 1 mm; el caso de capacidad usa margen 5 mm y separación 3 mm. Prioridad retazo → plancha.

## Diagnósticos — comparación FF/BF/WF/HF

In [1]:
# Importamos json para leer la traza que dejó el benchmark diagnóstico real.
import json
# Importamos Path para ubicar el repositorio desde la carpeta de ejecución.
from pathlib import Path
# Buscamos entre la carpeta actual y sus padres el artefacto diagnóstico del repositorio.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_trace.json").exists())
# Abrimos el archivo JSON con los placements, métricas y tiempos efectivamente medidos.
diagnostic = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_trace.json").read_text(encoding="utf-8"))
# Escribimos los títulos de la tabla breve para lectura directa.
print("Caso | Clase | opciones pocket | anclajes | componentes | usados | deteccion ms | placement ms | total ms | ratio")
# Recorremos los cuatro casos congelados en el orden de la evidencia.
for case in diagnostic["cases"]:
    # Tomamos el resultado del clasificador predeclarado.
    category = case["classification"]["category"]
    # Tomamos los contadores y el runtime total de Hole Filling.
    details = case["hole_filling"]["diagnostics"]
    # Imprimimos una fila con valores de la corrida guardada.
    print(f"{case['case_id']} | {category} | {details['pocket_options_by_decision']} | {details['geometric_positions_evaluated']} | {details['pocket_components_detected']} | {details['pockets_used']} | {details['pocket_detection_ms']:.2f} | {details['placement_search_ms']:.2f} | {details['total_ms']:.2f} | {case['classification']['runtime_ratio_vs_fastest_baseline']:.2f}x")


Caso | Clase | opciones pocket | anclajes | componentes | usados | deteccion ms | placement ms | total ms | ratio
HF-DIAGNOSTIC-001 | B | [3] | 839 | 3 | 1 | 0.89 | 9.43 | 14.09 | 4.60x
HF-DIAGNOSTIC-002 | B | [12] | 8532 | 12 | 1 | 7.22 | 11.07 | 21.68 | 5.85x
HF-DIAGNOSTIC-003 | B | [3] | 839 | 3 | 1 | 0.59 | 9.31 | 13.46 | 3.85x
HF-DIAGNOSTIC-004 | A | [0] | 19 | 0 | 0 | 0.02 | 11.25 | 13.13 | 1.08x


**Interpretación:** HF-001/002/003 son clase B: usan un pocket y cambian el patrón con métricas empatadas. HF-004 es clase A y recurre a FF; el control no inventa un pocket.

![HF-001: material y anclajes](../hole_filling/figures/HF-DIAGNOSTIC-001-physical.svg)

![HF-001: mapa de anclajes](../hole_filling/figures/HF-DIAGNOSTIC-001-used-01-anchors.svg)

![HF-002: anclajes triangulares](../hole_filling/figures/HF-DIAGNOSTIC-002-used-01-anchors.svg)

![HF-003: anclajes para convexos](../hole_filling/figures/HF-DIAGNOSTIC-003-used-01-anchors.svg)

![HF-004: control negativo](../hole_filling/figures/HF-DIAGNOSTIC-004-anchors.svg)

## Diagnósticos — mapa de regiones y anclajes

Distingue huecos físicos de pockets definidos por las posiciones legales de cada pieza.

In [2]:
# Importamos json para abrir la evidencia independiente de detección.
import json
# Importamos Path para localizar el directorio de evidencia.
from pathlib import Path
# Buscamos el repositorio que contiene el JSON de detección raster.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_detection.json").exists())
# Leemos regiones físicas y componentes de anclajes calculadas antes del placement.
detection = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_detection.json").read_text(encoding="utf-8"))
# Mostramos los campos que separan hueco físico, pocket y costo de clasificación.
print("Caso | huecos fisicos | orientaciones | anclajes examinados | pockets | fisico ms | mapa ms | componentes ms")
# Recorremos cada caso de detección real.
for case in detection:
    # Imprimimos la medida física y la cantidad de estados de anclaje revisados.
    print(f"{case['case_id']} | {case['internal_holes']} | {case['candidate_orientations']} | {case['anchor_positions_examined']} | {case['pocket_components']} | {case['physical_detection_ms']:.3f} | {case['anchor_generation_ms']:.3f} | {case['anchor_component_detection_ms']:.3f}")


Caso | huecos fisicos | orientaciones | anclajes examinados | pockets | fisico ms | mapa ms | componentes ms
HF-DIAGNOSTIC-001 | 0 | 2 | 839 | 3 | 0.494 | 0.118 | 0.937
HF-DIAGNOSTIC-002 | 0 | 12 | 8532 | 12 | 0.699 | 1.118 | 7.020
HF-DIAGNOSTIC-003 | 0 | 2 | 839 | 3 | 0.451 | 0.092 | 0.598
HF-DIAGNOSTIC-004 | 0 | 15 | 19 | 0 | 0.075 | 0.080 | 0.018


**Interpretación:** Los cuatro layouts tienen cero huecos físicos cerrados. El mapa de anclajes sí encuentra pockets para círculo, triángulo y convexo; el control no encuentra ninguno.

## Benchmark TA-027 — SMALL, MEDIUM y COMPLEX

Lee los resultados oficiales congelados de FF, BF, WF y tres repeticiones HF por caso.

In [3]:
# Importamos json para leer el benchmark oficial que pasó por la puerta VIABLE.
import json
# Importamos Path para encontrar la traza dentro del repositorio.
from pathlib import Path
# Buscamos la raíz local que contiene el benchmark EXP.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_official_trace.json").exists())
# Leemos los resultados secuenciales FF/BF/WF/HF ya ejecutados.
official = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hole_filling_official_trace.json").read_text(encoding="utf-8"))
# Imprimimos encabezados para el resumen funcional.
print("Caso | Método | Clase HF | Colocadas | Planchas | Uso % | Merma mm² | Recuperable mm² | ms")
# Recorremos cada fixture oficial experimental.
for case in official["cases"]:
    # Combinamos baselines con la corrida HF resumida del mismo caso.
    runs = case["baselines"] + [case["hole_filling"]]
    # Recorremos las cuatro políticas con las métricas originales.
    for run in runs:
        # Leemos métricas y mostramos una fila sin recalcularlas.
        values = run["metrics"]
        # Imprimimos la fila usando los valores originales del benchmark.
        print(f"{case['case_id']} | {run['method']} | {case['classification']['category'] if run['method'] == 'Hole-Filling' else '-'} | {values['placed']}/{values['requested']} | {values['new_sheets_used']} | {values['utilization_pct']} | {values['waste_mm2']} | {values['recoverable_remnant_mm2']} | {run['total_ms']:.2f}")


Caso | Método | Clase HF | Colocadas | Planchas | Uso % | Merma mm² | Recuperable mm² | ms
EXP-SMALL-001 | FF | - | 3/3 | 1 | 60.000 | 0.00 | 8 | 11.06
EXP-SMALL-001 | BF | - | 3/3 | 1 | 60.000 | 0.00 | 8 | 10.97
EXP-SMALL-001 | WF | - | 3/3 | 1 | 60.000 | 0.00 | 8 | 13.04
EXP-SMALL-001 | Hole-Filling | A | 3/3 | 1 | 60.000 | 0.00 | 8 | 12.57
EXP-MEDIUM-001 | FF | - | 9/9 | 1 | 28.413 | 201.86 | 58 | 17.80
EXP-MEDIUM-001 | BF | - | 9/9 | 1 | 28.413 | 193.86 | 66 | 28.43
EXP-MEDIUM-001 | WF | - | 9/9 | 2 | 18.584 | 301.86 | 150 | 17.81
EXP-MEDIUM-001 | Hole-Filling | D | 9/9 | 1 | 28.413 | 201.86 | 58 | 87.53
EXP-COMPLEX-001 | FF | - | 24/24 | 3 | 29.598 | 746.72 | 184 | 89.39
EXP-COMPLEX-001 | BF | - | 24/24 | 3 | 29.598 | 746.72 | 184 | 95.69
EXP-COMPLEX-001 | WF | - | 24/24 | 4 | 25.309 | 886.72 | 268 | 91.15
EXP-COMPLEX-001 | Hole-Filling | E | 24/24 | 3 | 29.598 | 746.72 | 184 | 1325.45


**Interpretación:** SMALL empata con FF sin pockets (A); MEDIUM queda detrás de BF en merma y recuperable (D); COMPLEX no usa pockets y supera 10× el tiempo del baseline más rápido (E). Las tres repeticiones por caso conservan la firma funcional: **NO-GO para integrar esta implementación en TA-042**.

## Pedidos realistas — HF-REALISTIC-001

Pedido congelado de 15 unidades con círculos y rectángulos; lee los resultados guardados.

In [4]:
# Importamos json para leer la traza real del benchmark realista.
import json
# Importamos Path para localizar el repositorio desde la carpeta de trabajo.
from pathlib import Path
# Buscamos la raiz que contiene los resultados congelados de SP-004.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").exists())
# Leemos la traza completa que incluye baselines y tres repeticiones HF.
trace = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").read_text(encoding="utf-8"))
# Seleccionamos solo el caso explicado en esta sección.
case = next(item for item in trace["cases"] if item["case_id"] == "HF-REALISTIC-001")
# Mostramos la tabla funcional medida para cada método.
print("Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms")
# Recorremos FF, BF, WF y el resumen HF del mismo pedido.
for run in case["baselines"] + [case["hole_filling"]]:
    # Leemos las métricas oficiales ya calculadas por el motor.
    metrics = run["official_metrics"]
    # Imprimimos cada método sin recalcular métricas.
    print(f"{run['method']} | {metrics['placed_units']}/{metrics['requested_units']} | {metrics['remnants_used']} | {metrics['sheets_used']} | {metrics['utilization_pct']} | {metrics['waste_mm2']} | {metrics['recoverable_area_mm2']} | {metrics['runtime_ms']:.2f}")
# Leemos los contadores específicos de Hole Filling.
hf = case["hole_filling"]
# Imprimimos la decisión, el costo y el determinismo medidos.
print(f"Clase {case['classification']['category']} | pockets {hf['pockets_detected']}/{hf['pockets_used']} | piezas en pocket {hf['pieces_placed_in_pockets']} | opciones {hf['pocket_candidates_checked']} | detección ms {hf['pocket_detection_ms']:.2f} | placement ms {hf['pocket_placement_ms']:.2f} | fallback {hf['fallback_count']} | cambios de fuente {hf['source_switches']} | determinista {case['deterministic']}")
# Leemos las tres repeticiones HF guardadas para este mismo fixture.
repetitions = case["hf_repetitions"]
# Imprimimos cada runtime individual sin incluirlo en la firma funcional.
print("HF repeticiones runtime ms | " + " | ".join(f"{run['official_metrics']['runtime_ms']:.2f}" for run in repetitions))
# Recorremos cada placement HF con su decisión de pocket o fallback.
for placement in hf["placements"]:
    # Imprimimos identidad, fuente, giro, coordenadas y decisión.
    print(f"pieza {placement['piece_id']} | {placement['source']} | {placement['rotation_deg']} grados | ({placement['x_mm']}, {placement['y_mm']}) mm | {placement['decision']}")


Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms
FF | 15/15 | 1 | 1 | 65.745 | 49908.91 | 18738 | 368.67
BF | 15/15 | 1 | 1 | 65.745 | 49908.91 | 18738 | 385.44
WF | 15/15 | 1 | 1 | 65.745 | 49908.91 | 18738 | 355.81
HF | 15/15 | 1 | 1 | 65.745 | 49908.91 | 18738 | 96383.48
Clase E | pockets 0/0 | piezas en pocket 0 | opciones 0 | detección ms 94789.53 | placement ms 1515.89 | fallback 15 | cambios de fuente 3 | determinista True
HF repeticiones runtime ms | 95080.06 | 96383.48 | 103334.88
pieza 4 | RETAZO | 0 grados | (5, 5) mm | fallback
pieza 3 | PLANCHA | 0 grados | (5, 5) mm | fallback
pieza 5 | PLANCHA | 0 grados | (188, 5) mm | fallback
pieza 2 | PLANCHA | 0 grados | (152, 128) mm | fallback
pieza 1 | PLANCHA | 0 grados | (5, 187) mm | fallback
pieza 6 | RETAZO | 0 grados | (5, 148) mm | fallback
pieza 7 | RETAZO | 0 grados | (88, 148) mm | fallback
pieza 9 | RETAZO | 0 grados | (228, 5) mm | fallback
pieza 8 | RETAZO | 0 grados | (228, 

**Interpretación:** Los círculos no generan pockets utilizables. HF reproduce placements y métricas del baseline, con clasificación E por costo.

## Pedidos realistas — HF-REALISTIC-002

Pedido congelado de 18 unidades con polígonos convexos y rectángulos; lee los resultados guardados.

In [5]:
# Importamos json para leer la traza real del benchmark realista.
import json
# Importamos Path para localizar el repositorio desde la carpeta de trabajo.
from pathlib import Path
# Buscamos la raiz que contiene los resultados congelados de SP-004.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").exists())
# Leemos la traza completa que incluye baselines y tres repeticiones HF.
trace = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").read_text(encoding="utf-8"))
# Seleccionamos solo el caso explicado en esta sección.
case = next(item for item in trace["cases"] if item["case_id"] == "HF-REALISTIC-002")
# Mostramos la tabla funcional medida para cada método.
print("Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms")
# Recorremos FF, BF, WF y el resumen HF del mismo pedido.
for run in case["baselines"] + [case["hole_filling"]]:
    # Leemos las métricas oficiales ya calculadas por el motor.
    metrics = run["official_metrics"]
    # Imprimimos cada método sin recalcular métricas.
    print(f"{run['method']} | {metrics['placed_units']}/{metrics['requested_units']} | {metrics['remnants_used']} | {metrics['sheets_used']} | {metrics['utilization_pct']} | {metrics['waste_mm2']} | {metrics['recoverable_area_mm2']} | {metrics['runtime_ms']:.2f}")
# Leemos los contadores específicos de Hole Filling.
hf = case["hole_filling"]
# Imprimimos la decisión, el costo y el determinismo medidos.
print(f"Clase {case['classification']['category']} | pockets {hf['pockets_detected']}/{hf['pockets_used']} | piezas en pocket {hf['pieces_placed_in_pockets']} | opciones {hf['pocket_candidates_checked']} | detección ms {hf['pocket_detection_ms']:.2f} | placement ms {hf['pocket_placement_ms']:.2f} | fallback {hf['fallback_count']} | cambios de fuente {hf['source_switches']} | determinista {case['deterministic']}")
# Leemos las tres repeticiones HF guardadas para este mismo fixture.
repetitions = case["hf_repetitions"]
# Imprimimos cada runtime individual sin incluirlo en la firma funcional.
print("HF repeticiones runtime ms | " + " | ".join(f"{run['official_metrics']['runtime_ms']:.2f}" for run in repetitions))
# Recorremos cada placement HF con su decisión de pocket o fallback.
for placement in hf["placements"]:
    # Imprimimos identidad, fuente, giro, coordenadas y decisión.
    print(f"pieza {placement['piece_id']} | {placement['source']} | {placement['rotation_deg']} grados | ({placement['x_mm']}, {placement['y_mm']}) mm | {placement['decision']}")


Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms
FF | 18/18 | 1 | 1 | 43.490 | 57164.00 | 66536 | 1740.82
BF | 18/18 | 1 | 1 | 43.490 | 57164.00 | 66536 | 1795.23
WF | 18/18 | 1 | 1 | 43.490 | 57164.00 | 66536 | 1792.27
HF | 18/18 | 1 | 1 | 43.490 | 57164.00 | 66536 | 236794.84
Clase E | pockets 11/1 | piezas en pocket 1 | opciones 11 | detección ms 231987.67 | placement ms 3073.25 | fallback 17 | cambios de fuente 5 | determinista True
HF repeticiones runtime ms | 236794.84 | 309478.45 | 166733.25
pieza 9 | RETAZO | 0 grados | (5, 5) mm | fallback
pieza 10 | RETAZO | 0 grados | (128, 5) mm | fallback
pieza 1 | RETAZO | 0 grados | (138, 88) mm | fallback
pieza 2 | RETAZO | 90 grados | (215, 56) mm | fallback
pieza 12 | RETAZO | 0 grados | (5, 98) mm | fallback
pieza 11 | PLANCHA | 0 grados | (5, 5) mm | fallback
pieza 3 | PLANCHA | 0 grados | (108, 5) mm | fallback
pieza 4 | PLANCHA | 0 grados | (241, 5) mm | fallback
pieza 13 | PLANCHA | 0 gra

**Interpretación:** HF coloca la pieza 5 en un pocket y cambia el patrón; empata métricas con FF/BF/WF, pero su costo medido es muy alto.

![Placement pocket-first HF-REALISTIC-002](../hole_filling/figures/HF-REALISTIC-002-pocket-01.svg)

## Pedidos realistas — HF-REALISTIC-003

Pedido mixto congelado de 20 unidades; lee los resultados guardados.

In [6]:
# Importamos json para leer la traza real del benchmark realista.
import json
# Importamos Path para localizar el repositorio desde la carpeta de trabajo.
from pathlib import Path
# Buscamos la raiz que contiene los resultados congelados de SP-004.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").exists())
# Leemos la traza completa que incluye baselines y tres repeticiones HF.
trace = json.loads((ROOT / "docs/evidence/sprint-02/SP-004/hole-filling/sp004_hf_realistic_trace.json").read_text(encoding="utf-8"))
# Seleccionamos solo el caso explicado en esta sección.
case = next(item for item in trace["cases"] if item["case_id"] == "HF-REALISTIC-003")
# Mostramos la tabla funcional medida para cada método.
print("Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms")
# Recorremos FF, BF, WF y el resumen HF del mismo pedido.
for run in case["baselines"] + [case["hole_filling"]]:
    # Leemos las métricas oficiales ya calculadas por el motor.
    metrics = run["official_metrics"]
    # Imprimimos cada método sin recalcular métricas.
    print(f"{run['method']} | {metrics['placed_units']}/{metrics['requested_units']} | {metrics['remnants_used']} | {metrics['sheets_used']} | {metrics['utilization_pct']} | {metrics['waste_mm2']} | {metrics['recoverable_area_mm2']} | {metrics['runtime_ms']:.2f}")
# Leemos los contadores específicos de Hole Filling.
hf = case["hole_filling"]
# Imprimimos la decisión, el costo y el determinismo medidos.
print(f"Clase {case['classification']['category']} | pockets {hf['pockets_detected']}/{hf['pockets_used']} | piezas en pocket {hf['pieces_placed_in_pockets']} | opciones {hf['pocket_candidates_checked']} | detección ms {hf['pocket_detection_ms']:.2f} | placement ms {hf['pocket_placement_ms']:.2f} | fallback {hf['fallback_count']} | cambios de fuente {hf['source_switches']} | determinista {case['deterministic']}")
# Leemos las tres repeticiones HF guardadas para este mismo fixture.
repetitions = case["hf_repetitions"]
# Imprimimos cada runtime individual sin incluirlo en la firma funcional.
print("HF repeticiones runtime ms | " + " | ".join(f"{run['official_metrics']['runtime_ms']:.2f}" for run in repetitions))
# Recorremos cada placement HF con su decisión de pocket o fallback.
for placement in hf["placements"]:
    # Imprimimos identidad, fuente, giro, coordenadas y decisión.
    print(f"pieza {placement['piece_id']} | {placement['source']} | {placement['rotation_deg']} grados | ({placement['x_mm']}, {placement['y_mm']}) mm | {placement['decision']}")


Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms
FF | 20/20 | 1 | 1 | 74.329 | 60627.26 | 14948 | 573.58
BF | 20/20 | 1 | 1 | 74.329 | 60627.26 | 14948 | 566.28
WF | 20/20 | 1 | 1 | 74.329 | 60627.26 | 14948 | 561.55
HF | 19/20 | 1 | 1 | 71.101 | 77120.58 | 7958 | 238463.28
Clase E | pockets 70/6 | piezas en pocket 6 | opciones 70 | detección ms 236147.70 | placement ms 2131.74 | fallback 14 | cambios de fuente 6 | determinista True
HF repeticiones runtime ms | 240619.94 | 238463.28 | 236980.21
pieza 1 | RETAZO | 0 grados | (5, 5) mm | fallback
pieza 2 | PLANCHA | 0 grados | (5, 5) mm | fallback
pieza 3 | RETAZO | 90 grados | (228, 5) mm | fallback
pieza 5 | PLANCHA | 0 grados | (208, 5) mm | fallback
pieza 4 | PLANCHA | 0 grados | (355, 113) mm | fallback
pieza 9 | PLANCHA | 0 grados | (5, 138) mm | fallback
pieza 13 | PLANCHA | 70 grados | (155, 132) mm | pocket
pieza 14 | PLANCHA | 170 grados | (242, 153) mm | pocket
pieza 6 | PLANCHA | 0 gr

**Interpretación:** HF usa seis pockets, pero deja una pieza sin colocar y empeora uso, merma y recuperable. La clasificación negativa se conserva.

![Placement pocket-first HF-REALISTIC-003](../hole_filling/figures/HF-REALISTIC-003-pocket-01.svg)

Figuras adicionales: [`pocket 02`](../hole_filling/figures/HF-REALISTIC-003-pocket-02.svg), [`pocket 03`](../hole_filling/figures/HF-REALISTIC-003-pocket-03.svg), [`pocket 04`](../hole_filling/figures/HF-REALISTIC-003-pocket-04.svg), [`pocket 05`](../hole_filling/figures/HF-REALISTIC-003-pocket-05.svg), [`pocket 06`](../hole_filling/figures/HF-REALISTIC-003-pocket-06.svg).

## Experimento demostrativo de capacidad — HF-CAPABILITY-001

**Caso controlado para demostrar capacidad, no rendimiento general.** Verifica el fixture: raster 1 mm, giros 5°, margen 5 mm y separación 3 mm.

In [7]:
# Importamos hashlib para verificar los bytes del fixture congelado.
import hashlib
# Importamos statistics para resumir las tres duraciones HF.
import statistics
# Importamos sys para cargar los paquetes locales del repositorio.
import sys
# Importamos Path para ubicar casos, trazas y figuras.
from pathlib import Path
# Encontramos la raíz que contiene el fixture de capacidad.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "experiments/optimization/hole_filling/cases/HF-CAPABILITY-001.json").exists())
# Exponemos el backend real al intérprete del notebook.
sys.path.insert(0, str(ROOT / "backend"))
# Exponemos los experimentos del repositorio al intérprete.
sys.path.insert(0, str(ROOT))
# Importamos el lector del fixture y su validación de semillas.
from experiments.optimization.hole_filling.lab import read_case, prepare_case
# Importamos las cuatro heurísticas experimentales y las métricas originales.
from experiments.optimization.hole_filling.optimizer import run_baseline, run_hole_filling, metric_values
# Importamos la firma funcional usada por el benchmark definitivo.
from experiments.optimization.hole_filling.run_realistic_benchmark import _determinism_signature
# Importamos los renderizadores grises que usan placements reales.
from experiments.optimization.hole_filling.visualize import render_capability_comparison, render_pocket_placement
# Formamos la ruta al JSON congelado.
case_path = ROOT / "experiments/optimization/hole_filling/cases/HF-CAPABILITY-001.json"
# Verificamos que ningún byte del fixture cambió después de la congelación.
assert hashlib.sha256(case_path.read_bytes()).hexdigest() == "565a7bcd47a8de773f9f7e0c5804a35b618db1e32c0074d8e45eaeacf5d565e5"
# Leemos la misma configuración que usarán FF, BF, WF y HF.
case = read_case("HF-CAPABILITY-001")
# Validamos todas las piezas previas con SheetLayout.check/place antes de correr.
setup = prepare_case("HF-CAPABILITY-001")
# Mostramos la configuración compartida por todos los métodos.
print("Configuración:", case["config"], "| material:", case["material"])
# Recorremos retazo y plancha del fixture controlado.
for stock in case["stocks"]:
    # Mostramos fuente, identificador y dimensiones sin modificarlos.
    print("Stock:", stock["source"], stock["material_id"], stock["geometry"])
# Recorremos cada pieza para distinguir las previas de las pendientes.
for piece in case["pieces"]:
    # Mostramos identidad, geometría y estado inicial de cada pieza.
    print("Pieza:", piece["piece_id"], piece["role"], piece["geometry"])


Configuración: {'resolution_mm': 1, 'angular_step_deg': 5, 'margin_mm': 5, 'separation_mm': 3} | material: {'glass_type_id': 1, 'thickness_mm': 6}
Stock: RETAZO 8201 {'type': 'RECTANGULO', 'width_mm': 274, 'height_mm': 280}
Stock: PLANCHA 9201 {'type': 'RECTANGULO', 'width_mm': 274, 'height_mm': 280}
Pieza: 1 seed {'type': 'CIRCUNFERENCIA', 'radius_mm': 24}
Pieza: 2 seed {'type': 'RECTANGULO', 'width_mm': 48, 'height_mm': 24}
Pieza: 3 seed {'type': 'RECTANGULO', 'width_mm': 48, 'height_mm': 24}
Pieza: 4 seed {'type': 'RECTANGULO', 'width_mm': 24, 'height_mm': 96}
Pieza: 5 seed {'type': 'RECTANGULO', 'width_mm': 24, 'height_mm': 96}
Pieza: 6 seed {'type': 'RECTANGULO', 'width_mm': 96, 'height_mm': 24}
Pieza: 7 seed {'type': 'RECTANGULO', 'width_mm': 194, 'height_mm': 36}
Pieza: 8 seed {'type': 'RECTANGULO', 'width_mm': 43, 'height_mm': 192}
Pieza: 9 seed {'type': 'RECTANGULO', 'width_mm': 43, 'height_mm': 192}
Pieza: 10 seed {'type': 'RECTANGULO', 'width_mm': 264, 'height_mm': 35}
Pieza

**Interpretación:** El fixture muestra 16 placements previos y dos piezas pendientes; fuentes, geometría y parámetros son iguales para los cuatro métodos.

## Experimento demostrativo — First Fit

Ejecuta FF sobre el mismo fixture congelado.

In [8]:
# Ejecutamos el método FF con las semillas validadas y el fixture congelado.
ff_run = run_baseline("HF-CAPABILITY-001", "FF")
# Leemos las métricas calculadas por el motor para esta corrida.
values = metric_values(ff_run)
# Mostramos cobertura, planchas, utilización, merma, recuperable y runtime real.
print("FF:", values["placed"], "/", values["requested"], "| planchas", values["new_sheets_used"], "| uso", values["utilization_pct"], "| merma", values["waste_mm2"], "| recuperable", values["recoverable_remnant_mm2"], "| ms", round(ff_run["total_ms"], 2))
# Recorremos los placements producidos por el método.
for placement in ff_run["result"].placements:
    # Filtramos las piezas pendientes para distinguirlas del layout previo común.
    if placement.piece.piece_id in (11, 12):
        # Imprimimos pieza, fuente, giro y coordenadas efectivamente usadas.
        print("Pieza", placement.piece.piece_id, placement.stock.source.value, str(placement.rotation_deg) + "°", "(", placement.x_mm, ",", placement.y_mm, ") mm")


FF: 18 / 18 | planchas 1 | uso 33.787 | merma 37516.56 | recuperable 64080 | ms 61.23
Pieza 11 RETAZO 0° ( 5 , 5 ) mm
Pieza 12 PLANCHA 0° ( 5 , 5 ) mm


**Interpretación:** FF coloca el inserto en la franja exterior y necesita la plancha para el círculo posterior.

## Experimento demostrativo — Best Fit

Ejecuta BF sobre el mismo fixture congelado.

In [9]:
# Ejecutamos el método BF con las semillas validadas y el fixture congelado.
bf_run = run_baseline("HF-CAPABILITY-001", "BF")
# Leemos las métricas calculadas por el motor para esta corrida.
values = metric_values(bf_run)
# Mostramos cobertura, planchas, utilización, merma, recuperable y runtime real.
print("BF:", values["placed"], "/", values["requested"], "| planchas", values["new_sheets_used"], "| uso", values["utilization_pct"], "| merma", values["waste_mm2"], "| recuperable", values["recoverable_remnant_mm2"], "| ms", round(bf_run["total_ms"], 2))
# Recorremos los placements producidos por el método.
for placement in bf_run["result"].placements:
    # Filtramos las piezas pendientes para distinguirlas del layout previo común.
    if placement.piece.piece_id in (11, 12):
        # Imprimimos pieza, fuente, giro y coordenadas efectivamente usadas.
        print("Pieza", placement.piece.piece_id, placement.stock.source.value, str(placement.rotation_deg) + "°", "(", placement.x_mm, ",", placement.y_mm, ") mm")


BF: 18 / 18 | planchas 1 | uso 33.787 | merma 37516.56 | recuperable 64080 | ms 66.2
Pieza 11 RETAZO 0° ( 5 , 5 ) mm
Pieza 12 PLANCHA 0° ( 5 , 5 ) mm


**Interpretación:** BF elige la misma franja para el inserto y necesita la plancha para el círculo posterior.

## Experimento demostrativo — Worst Fit

Ejecuta WF sobre el mismo fixture congelado.

In [10]:
# Ejecutamos el método WF con las semillas validadas y el fixture congelado.
wf_run = run_baseline("HF-CAPABILITY-001", "WF")
# Leemos las métricas calculadas por el motor para esta corrida.
values = metric_values(wf_run)
# Mostramos cobertura, planchas, utilización, merma, recuperable y runtime real.
print("WF:", values["placed"], "/", values["requested"], "| planchas", values["new_sheets_used"], "| uso", values["utilization_pct"], "| merma", values["waste_mm2"], "| recuperable", values["recoverable_remnant_mm2"], "| ms", round(wf_run["total_ms"], 2))
# Recorremos los placements producidos por el método.
for placement in wf_run["result"].placements:
    # Filtramos las piezas pendientes para distinguirlas del layout previo común.
    if placement.piece.piece_id in (11, 12):
        # Imprimimos pieza, fuente, giro y coordenadas efectivamente usadas.
        print("Pieza", placement.piece.piece_id, placement.stock.source.value, str(placement.rotation_deg) + "°", "(", placement.x_mm, ",", placement.y_mm, ") mm")


WF: 18 / 18 | planchas 1 | uso 33.787 | merma 37516.56 | recuperable 64080 | ms 62.01
Pieza 11 RETAZO 0° ( 5 , 5 ) mm
Pieza 12 PLANCHA 0° ( 5 , 5 ) mm


**Interpretación:** WF también usa la franja exterior para el inserto; el círculo posterior queda en la plancha.

## Experimento demostrativo — Hole Filling

Ejecuta HF tres veces sobre el mismo fixture congelado.

In [11]:
# Creamos una lista para conservar las tres corridas independientes de HF.
hf_runs = []
# Ejecutamos HF tres veces de forma serial sobre el mismo JSON congelado.
for repetition in range(1, 4):
    # Guardamos cada corrida completa para comparar placements y métricas.
    hf_runs.append(run_hole_filling("HF-CAPABILITY-001"))
    # Mostramos el runtime individual sin usarlo como parte de la firma.
    print("HF repetición", repetition, "| ms", round(hf_runs[-1]["total_ms"], 2))
# Construimos las firmas funcionales que excluyen el tiempo.
hf_signatures = [_determinism_signature(run) for run in hf_runs]
# Comprobamos que las tres firmas se repitieron exactamente.
hf_deterministic = len(set(hf_signatures)) == 1
# Mostramos PASS o FAIL de determinismo.
print("Determinismo:", "PASS" if hf_deterministic else "FAIL")
# Conservamos la primera corrida para sus placements y figuras.
hf_run = hf_runs[0]
# Leemos las métricas originales de Hole Filling.
hf_values = metric_values(hf_run)
# Imprimimos cobertura, uso, merma y fuentes de la primera corrida.
print("HF:", hf_values["placed"], "/", hf_values["requested"], "| planchas", hf_values["new_sheets_used"], "| uso", hf_values["utilization_pct"], "| merma", hf_values["waste_mm2"], "| recuperable", hf_values["recoverable_remnant_mm2"])
# Mostramos componentes detectados, pockets usados y fallbacks.
print("Pockets:", hf_run["diagnostics"]["pocket_components_detected"], "| usados:", hf_run["diagnostics"]["pockets_used"], "| fallback:", hf_run["diagnostics"]["fallback_placements"] + hf_run["diagnostics"]["fallback_unplaced"])
# Recorremos las piezas pendientes colocadas por HF.
for placement in hf_run["result"].placements:
    # Separamos las dos piezas pendientes de las dieciséis semillas.
    if placement.piece.piece_id in (11, 12):
        # Etiquetamos el placement según el pocket seleccionado.
        decision = "pocket" if any(item["piece_id"] == placement.piece.piece_id for item in hf_run["diagnostics"]["selected_pockets"]) else "fallback"
        # Mostramos pieza, fuente, giro, posición y decisión.
        print("Pieza", placement.piece.piece_id, placement.stock.source.value, str(placement.rotation_deg) + "°", "(", placement.x_mm, ",", placement.y_mm, ") mm", decision)


HF repetición 1 | ms 263.89
HF repetición 2 | ms 284.81
HF repetición 3 | ms 267.06
Determinismo: PASS
HF: 18 / 18 | planchas 0 | uso 67.575 | merma 22410.56 | recuperable 2466
Pockets: 7 | usados: 1 | fallback: 1
Pieza 11 RETAZO 0° ( 51 , 191 ) mm pocket
Pieza 12 RETAZO 0° ( 5 , 5 ) mm fallback


**Interpretación:** HF detecta siete componentes candidatos y coloca el inserto P11 en un pocket del retazo. Así deja sitio exterior para P12.

## Experimento demostrativo — comparación visual

Dibuja las colocaciones y el mapa de anclajes del caso controlado.

In [12]:
# Indicamos la carpeta de figuras dentro del experimento aislado.
figure_dir = ROOT / "experiments/optimization/hole_filling/figures"
# Dibujamos los cuatro layouts finales a partir de las corridas vivas.
render_capability_comparison({"FF": ff_run, "BF": bf_run, "WF": wf_run, "HF": hf_run}, figure_dir / "HF-CAPABILITY-001-comparison.svg")
# Extraemos el estado inmediatamente anterior al placement pocket de HF.
layout, anchor_map, selected_anchor, pending_mask = hf_run["pocket_placement_contexts"][0]
# Dibujamos layout previo, anclajes, pocket, inserto y posición elegida.
render_pocket_placement(layout, anchor_map, selected_anchor, pending_mask, figure_dir / "HF-CAPABILITY-001-pocket-stage.svg", "HF-CAPABILITY-001 | antes y durante el pocket")
# Mostramos las rutas relativas para que el output no dependa del equipo local.
figure_paths = (figure_dir / "HF-CAPABILITY-001-comparison.svg", figure_dir / "HF-CAPABILITY-001-pocket-stage.svg")
print("Figuras:", " | ".join(path.relative_to(ROOT).as_posix() for path in figure_paths))


Figuras: experiments/optimization/hole_filling/figures/HF-CAPABILITY-001-comparison.svg | experiments/optimization/hole_filling/figures/HF-CAPABILITY-001-pocket-stage.svg


**Interpretación:** El pocket aparece solo en HF; FF/BF/WF usan la plancha para P12. El segundo mapa separa anclajes exteriores del componente pocket.

![FF, BF, WF y HF: layouts finales](../hole_filling/figures/HF-CAPABILITY-001-comparison.svg)

![Layout previo, pocket y pieza pendiente](../hole_filling/figures/HF-CAPABILITY-001-pocket-stage.svg)

## Experimento demostrativo — métricas

Compara cobertura, fuentes, uso, merma, recuperable y tiempo medidos.

In [13]:
# Mostramos las columnas oficiales para comparar los cuatro métodos.
print("Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms")
# Recorremos FF, BF, WF y HF en el mismo orden que el benchmark definitivo.
for method, run in (("FF", ff_run), ("BF", bf_run), ("WF", wf_run), ("HF", hf_run)):
    # Leemos las métricas calculadas por el mismo motor para cada corrida.
    values = metric_values(run)
    # Contamos las unidades físicas de retazo usadas por los placements.
    remnants = len({(p.stock.material_id, p.stock.physical_unit_index) for p in run["result"].placements if p.stock.source.value == "RETAZO"})
    # Resumimos el runtime HF con sus tres repeticiones; los demás tienen una.
    duration = statistics.median(item["total_ms"] for item in hf_runs) if method == "HF" else run["total_ms"]
    # Imprimimos la fila sin alterar las fórmulas oficiales.
    print(method, "|", str(values["placed"]) + "/" + str(values["requested"]), "|", remnants, "|", values["new_sheets_used"], "|", values["utilization_pct"], "|", values["waste_mm2"], "|", values["recoverable_remnant_mm2"], "|", round(duration, 2))


Método | colocadas | retazos | planchas | uso % | merma mm² | recuperable mm² | runtime ms
FF | 18/18 | 1 | 1 | 33.787 | 37516.56 | 64080 | 61.23
BF | 18/18 | 1 | 1 | 33.787 | 37516.56 | 64080 | 66.2
WF | 18/18 | 1 | 1 | 33.787 | 37516.56 | 64080 | 62.01
HF | 18/18 | 1 | 0 | 67.575 | 22410.56 | 2466 | 267.06


**Interpretación:** Todos colocan 18/18. FF/BF/WF usan retazo y plancha; HF solo retazo: uso 33.787% → 67.575%, merma 37516.56 → 22410.56 mm² y recuperable 64080 → 2466 mm². La salida de esta ejecución del notebook registra 267.06 ms de mediana HF y 61.23/66.20/62.01 ms para FF/BF/WF. La traza y el CSV del benchmark final registran 262.99 ms y 63.05/62.32/62.81 ms, valores usados en la SPEC y evidencia final. El caso dirigido no estima el rendimiento general.